In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

# glom()

- Coalesces all elements within each partition into a list
- Useful for understanding data distribution across partitions
- Syntax: rdd.glom()

In [88]:
rdd1 = spark.sparkContext.parallelize([1, 2, 3, 4, 5, 6], 3)
print(f"Original RDD: {rdd1.collect()}")
print(f"After glom(): {rdd1.glom().collect()}")

Original RDD: [1, 2, 3, 4, 5, 6]
After glom(): [[1, 2], [3, 4], [5, 6]]


# RDD Repartition

It is used to increase or decrease the number of partitions in an RDD.
Can improve performance based on your cluster resources.

### Key Points:
- repartition(numPartitions): Creates a new RDD with exactly `numPartitions` partitions
- Uses a shuffle operation (expensive)
- Can increase or decrease partitions
- Data is distributed evenly across partitions using hash partitioning


> **Note:** For only decreasing partitions, use coalesce() instead (avoids full shuffle)






In [51]:
# Create RDD from sample data
data = range(1, 51)
rdd = spark.sparkContext.parallelize(data, 5)  # Initially  partitions
rdd.getNumPartitions()

5

In [55]:
# Data distribution
rdd.glom().map(len).collect()

[10, 10, 10, 10, 10]

In [58]:
# Repartition to 10 partitions (increase)
repartitioned_rdd = rdd.repartition(10)
repartitioned_rdd.getNumPartitions()

10

In [59]:
# Data distribution
repartitioned_rdd.glom().map(len).collect()

[0, 0, 0, 0, 20, 0, 0, 0, 10, 20]

In [56]:
# Repartition to 1 partition (decrease)
single_partition_rdd = rdd.repartition(1)
single_partition_rdd.getNumPartitions()

1

In [60]:
# Data distribution
single_partition_rdd.glom().map(len).collect()

[50]

# RDD Coalesce

Coalesce is used to reduce the number of partitions in an RDD efficiently.  
It avoids a full shuffle and is optimized for decreasing partitions.

### Key Points:
- Reduces partitions without shuffling (by default)
- More efficient than repartition() when decreasing partitions
- Cannot increase partitions beyond current count (unless shuffle=True)
- Use when you want to optimize performance by reducing parallelism

In [61]:
data = range(1, 101)
rdd = spark.sparkContext.parallelize(data, 10)  # Create RDD with 10 partitions
rdd.getNumPartitions()

10

In [63]:
# Data distribution
rdd.glom().map(len).collect()

[10, 10, 10, 10, 10, 10, 10, 10, 10, 10]

In [62]:
# Use coalesce to reduce partitions from 10 to 2
coalesced_rdd = rdd.coalesce(2)
coalesced_rdd.getNumPartitions()

2

In [64]:
# Data distribution
coalesced_rdd.glom().map(len).collect()

[50, 50]

# RDD repartitionAndSortWithinPartitions

Repartitions an RDD and sorts records by key within each partition in a single
efficient operation during the shuffle.

### Key Points:
- Works only on key-value pair RDDs
- More efficient than separate repartition() + sortByKey()
- Sorting is per-partition, not global

### Example 1: Basic usage with key-value pairs

In [83]:
data = [("apple", 5), ("banana", 2), ("apple", 3), ("cherry", 8), ("banana", 1), ("cherry", 4), ("apple", 7)]
rdd = spark.sparkContext.parallelize(data, 2)

In [84]:
# Repartition into 3 partitions and sort by key within each partition
sorted_rdd = rdd.repartitionAndSortWithinPartitions(numPartitions=3, ascending=True)
sorted_rdd.glom().collect()

[[],
 [],
 [('apple', 5),
  ('apple', 3),
  ('apple', 7),
  ('banana', 2),
  ('banana', 1),
  ('cherry', 8),
  ('cherry', 4)]]

### Example 2: Custom partitioner with hash function

In [85]:
sales_data = [("NY", 100), ("CA", 200), ("NY", 150), ("TX", 300), ("CA", 250), ("TX", 100), ("NY", 50), ("CA", 180)]
sales_rdd = spark.sparkContext.parallelize(sales_data, 2)

In [86]:
sorted_sales = sales_rdd.repartitionAndSortWithinPartitions(numPartitions=2, partitionFunc=lambda x: hash(x) % 2, ascending=True)
sorted_sales.glom().collect()

[[('CA', 200), ('CA', 250), ('CA', 180), ('NY', 100), ('NY', 150), ('NY', 50)],
 [('TX', 300), ('TX', 100)]]

# Difference between `repartition()` and `coalesce()`

1. repartition():
    - Can increase or decrease the number of partitions
    - Performs a full shuffle of data across the cluster
    - More expensive operation due to network I/O
    - Results in evenly distributed data across partitions
    - Use when you need to increase partitions or need even distribution

2. coalesce():
    - Can only decrease the number of partitions
    - Avoids full shuffle by moving data to fewer partitions
    - More efficient and faster than repartition
    - May result in uneven data distribution
    - Use when you need to reduce partitions and can accept uneven distribution